# تجارت‌یار نسخه ۴.۰ — پیش‌نمایش شاخه (بدون ادغام)

این نوتبوک مخصوص **تست سریع نسخهٔ جدید** است و به‌صورت خودکار از شاخهٔ `arena/01a01549-tejaratyar` اجرا می‌شود.

۱. **Runtime → Run all**  
۲. منتظر بمان تا لینک `trycloudflare.com` چاپ شود  
۳. همان لینک را باز کن یا برای دیگران بفرست  
۴. تا زمان استفاده، تب Colab و Runtime را نبند

> این لینک موقت است. برای نسخهٔ پایدار پس از ادغام، از `TejaratYar.ipynb` (شاخهٔ main) استفاده کنید.


In [ ]:
import os, re, sys, time, subprocess, urllib.request
from IPython.display import display, HTML

REPO = "https://github.com/Setayesh-Jafari/tejaratyar.git"
REPO_ROOT = "/content/tejaratyar"
ROOT = REPO_ROOT

# شاخه‌ای که باید اجرا شود. پیش‌فرض "main" است. برای تست یک شاخهٔ دیگر پیش از ادغام،
# مقدار را به نام همان شاخه تغییر بده (مثلاً TY_BRANCH = "arena/01a01549-tejaratyar").
TY_BRANCH = os.environ.get("TY_BRANCH", "arena/01a01549-tejaratyar").strip()

os.chdir("/content")
if not os.path.exists(os.path.join(REPO_ROOT, ".git")):
    if TY_BRANCH and TY_BRANCH != "main":
        subprocess.run(["git", "clone", "--depth", "1", "--branch", TY_BRANCH, REPO, REPO_ROOT], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO, REPO_ROOT], check=True)
else:
    subprocess.run(["git", "-C", REPO_ROOT, "pull", "--ff-only"], check=False)

if TY_BRANCH and TY_BRANCH != "main":
    subprocess.run(["git", "-C", REPO_ROOT, "fetch", "--depth", "1", "origin", TY_BRANCH], check=False)
    subprocess.run(["git", "-C", REPO_ROOT, "checkout", "-B", TY_BRANCH, f"origin/{TY_BRANCH}"], check=False)

if not os.path.exists(os.path.join(ROOT, "agent", "pipeline.py")):
    raise RuntimeError("فایل‌های اصلی تجارت‌یار در ریشه مخزن پیدا نشدند. محتویات بسته آماده را در ریشه مخزن قرار دهید.")

os.chdir(ROOT)
print("نسخه پروژه:", ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# اجرای دوباره سلول نباید چند سرور روی یک پورت باقی بگذارد.
subprocess.run("pkill -f 'gunicorn app:app' || true", shell=True)
time.sleep(1)
env = os.environ.copy()
env.update({"PORT": "5000", "MAX_ACTIVE_JOBS": "3", "JOB_TTL_HOURS": "24", "APP_TIMEZONE": "Asia/Tehran"})
logf = open("/tmp/tejaratyar.log", "w")
server = subprocess.Popen(
    [sys.executable, "-m", "gunicorn", "app:app", "--bind", "0.0.0.0:5000",
     "--workers", "1", "--threads", "8", "--timeout", "300"],
    cwd=ROOT, env=env, stdout=logf, stderr=subprocess.STDOUT,
)

for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:5000/health", timeout=2)
        break
    except Exception:
        if server.poll() is not None:
            logf.flush(); print(open("/tmp/tejaratyar.log").read())
            raise RuntimeError("سرور متوقف شد؛ لاگ بالا را بررسی کن.")
        time.sleep(1)
else:
    logf.flush(); print(open("/tmp/tejaratyar.log").read())
    raise RuntimeError("سرور در ۶۰ ثانیه آماده نشد.")

subprocess.run(["wget", "-q", "-O", "/tmp/cloudflared",
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
subprocess.run(["chmod", "+x", "/tmp/cloudflared"], check=True)
subprocess.run("pkill -f '/tmp/cloudflared tunnel' || true", shell=True)
time.sleep(1)
tunnel = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:5000", "--no-autoupdate"],
    stderr=subprocess.PIPE, text=True,
)
public = None
start = time.time()
while time.time() - start < 90:
    line = tunnel.stderr.readline()
    if not line and tunnel.poll() is not None:
        break
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if m:
        public = m.group(0)
        html = f'<div dir="rtl" style="font-family:Tahoma;padding:18px;border-radius:14px;background:#e7f1fb;border:1px solid #0b5cab"><div style="font-size:14px;color:#084a8a;margin-bottom:8px">تجارت‌یار نسخه حرفه‌ای آماده است. این تب را نبند.</div><a href="{public}" target="_blank" style="font-size:22px;font-weight:700;color:#0b5cab">{public}</a></div>'
        display(HTML(html))
        print("SHARE_URL", public)
        break
if not public:
    raise RuntimeError("لینک تونل ساخته نشد؛ سلول را دوباره اجرا کن و لاگ را بررسی کن.")

# زنده نگه داشتن تونل و سلول
tunnel.wait()
